In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("filter-data")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/24 15:05:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles_extended.csv"))


df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                null|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                null|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [11]:
df_countries_split = (df.withColumn("country",split(col("country"), ', ')))
df_countries = (df_countries_split.withColumn("country",explode(col("country"))))

df_countries.show()

+-------+-------+--------------------+-------------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|           director|                cast|       country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+-------------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|    Kirsten Johnson|                null| United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|               null|Ama Qamata, Khosi...|  South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s5|TV Show|        Kota Factory|               null|Mayur More, 

In [14]:
grouped_df = df_countries.groupBy("country")

count_df = grouped_df.count().orderBy("count", ascending=False)

count_df.show()

+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 3675|
|         India| 1046|
|United Kingdom|  803|
|        Canada|  445|
|        France|  392|
|         Japan|  318|
|   South Korea|  231|
|         Spain|  230|
|       Germany|  224|
|        Mexico|  169|
|         China|  162|
|     Australia|  160|
|         Egypt|  117|
|        Turkey|  113|
|     Hong Kong|  105|
|       Nigeria|  101|
|         Italy|  100|
|        Brazil|   97|
|     Argentina|   91|
|       Belgium|   90|
+--------------+-----+
only showing top 20 rows



In [15]:
spark.stop()